In [3]:
import pandas as pd
from pathlib import Path

Pathway 

In [4]:
press_conf_fed = Path('/Users/lorenzouberti/Desktop/cvbnjkl')
press_conf = pd.read_excel( press_conf_fed / "fed_statements_presconf_WITH_LLM_LABELS_2021.xlsx")
press_conf_ecb = Path('/Users/lorenzouberti/Desktop/cvbnjkl')

FileNotFoundError: [Errno 2] No such file or directory: '/Users/lorenzouberti/Desktop/cvbnjkl/fed_statements_presconf_WITH_LLM_LABELS_2021.xlsx'

Analysis

In [ ]:
press_conf.columns

Index(['source', 'orig_row', 'id', 'raw_text', 'doc_type', 'clean_text',
       'sentence', 'Dissagreement', 'label', 'Unnamed: 9', 'llm_majority',
       'claude-haiku__p1_analyst_direct', 'claude-haiku__p2_analyst_rubric',
       'claude-haiku__p3_analyst_rules', 'claude-haiku__p4_analyst_json',
       'claude-haiku__p5_analyst_fewshot', 'claude-sonnet__p1_analyst_direct',
       'claude-sonnet__p2_analyst_rubric', 'claude-sonnet__p3_analyst_rules',
       'claude-sonnet__p4_analyst_json', 'claude-sonnet__p5_analyst_fewshot',
       'deepseek-chat__p1_analyst_direct', 'deepseek-chat__p2_analyst_rubric',
       'deepseek-chat__p3_analyst_rules', 'deepseek-chat__p4_analyst_json',
       'deepseek-chat__p5_analyst_fewshot',
       'deepseek-reasoner__p1_analyst_direct',
       'deepseek-reasoner__p2_analyst_rubric',
       'deepseek-reasoner__p3_analyst_rules',
       'deepseek-reasoner__p4_analyst_json',
       'deepseek-reasoner__p5_analyst_fewshot',
       'gpt-5-mini__p1_analyst_dir

In [ ]:
"""
Analyze where each LLM/prompt combination disagrees with the true (human) label,
then summarize — per model (pooled across its 5 prompts) — which category
(the 'Dissagreement' column, e.g. Economic Status, Labor, Fed Expectations, etc.)
it misclassifies most often.

Expects a dataframe `press_conf` with columns like:
    'label'                              -> ground-truth class (D/N/H)
    'Dissagreement'                      -> category tag for the sentence
    '<model>__<prompt>'                  -> model's predicted class for that prompt
"""

import pandas as pd
from pathlib import Path

# ---------------------------------------------------------------
# 1. Load data
# ---------------------------------------------------------------
press_conf_fed = Path(
    '/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Github Repos/Project/'
    'BdF-Project/Extension LLM Classification Task/Fed Side/Excel Sentences Results'
)
press_conf = pd.read_excel(
    press_conf_fed / "fed_statements_presconf_WITH_LLM_LABELS_2021.xlsx"
)

TRUE_LABEL_COL = "label"
CATEGORY_COL = "Dissagreement"

# ---------------------------------------------------------------
# 2. Identify all model/prompt prediction columns automatically
#    (anything with the "<model>__<prompt>" naming pattern)
# ---------------------------------------------------------------
pred_cols = [c for c in press_conf.columns if "__" in c]

# Keep only rows that actually have a ground-truth label
df = press_conf.dropna(subset=[TRUE_LABEL_COL]).copy()

# ---------------------------------------------------------------
# 3. Build a long/tidy "miss table": one row per (sentence, model, prompt)
#    where the prediction != true label
# ---------------------------------------------------------------
records = []
for col in pred_cols:
    model, prompt = col.split("__", 1)
    sub = df[[TRUE_LABEL_COL, CATEGORY_COL, col]].copy()
    sub = sub.rename(columns={col: "pred"})
    sub["model"] = model
    sub["prompt"] = prompt
    sub["sentence_id"] = df.index  # or df['id'] if you prefer a stable id
    records.append(sub)

long_df = pd.concat(records, ignore_index=True)

# Drop rows where the model didn't produce a prediction at all (NaN)
long_df = long_df.dropna(subset=["pred"])

# Flag misses: prediction != true label
long_df["is_miss"] = long_df["pred"].str.strip().str.upper() != long_df[
    TRUE_LABEL_COL
].str.strip().str.upper()

misses = long_df[long_df["is_miss"]].copy()

# ---------------------------------------------------------------
# 4. Per model x prompt: how many misses, and by category
# ---------------------------------------------------------------
miss_counts_by_model_prompt = (
    misses.groupby(["model", "prompt", CATEGORY_COL])
    .size()
    .rename("n_misses")
    .reset_index()
    .sort_values(["model", "prompt", "n_misses"], ascending=[True, True, False])
)

print("=== Misses by model, prompt, and category ===")
print(miss_counts_by_model_prompt.to_string(index=False))

# ---------------------------------------------------------------
# 5. Per model, POOLED across all 5 prompts:
#    which category does each model misclassify the most?
# ---------------------------------------------------------------
miss_counts_by_model = (
    misses.groupby(["model", CATEGORY_COL])
    .size()
    .rename("n_misses")
    .reset_index()
)

# Top miscategorized category per model
top_category_per_model = (
    miss_counts_by_model.sort_values(["model", "n_misses"], ascending=[True, False])
    .groupby("model")
    .head(1)
    .reset_index(drop=True)
)

print("\n=== Top miscategorized category per model (pooled across prompts) ===")
print(top_category_per_model.to_string(index=False))

# Full ranked breakdown (all categories, ranked, per model) — often more useful
ranked_by_model = (
    miss_counts_by_model.sort_values(["model", "n_misses"], ascending=[True, False])
    .reset_index(drop=True)
)
print("\n=== Full ranked category miss-breakdown per model ===")
print(ranked_by_model.to_string(index=False))

# ---------------------------------------------------------------
# 5b. Same as above but normalized: error RATE within each category
#     (misses in that category / total predictions made in that category)
#     This matters because some categories (e.g. Economic Status) simply
#     have way more sentences than others (e.g. Key Words), so raw miss
#     counts alone are misleading.
# ---------------------------------------------------------------
# Total predictions made per model, per category (denominator)
total_by_model_cat = (
    long_df.groupby(["model", CATEGORY_COL])
    .size()
    .rename("n_total")
    .reset_index()
)

# Merge misses with totals to compute rate
rate_by_model_cat = miss_counts_by_model.merge(
    total_by_model_cat, on=["model", CATEGORY_COL], how="right"
)
rate_by_model_cat["n_misses"] = rate_by_model_cat["n_misses"].fillna(0)
rate_by_model_cat["error_rate"] = (
    rate_by_model_cat["n_misses"] / rate_by_model_cat["n_total"]
)

rate_by_model_cat = rate_by_model_cat.sort_values(
    ["model", "error_rate"], ascending=[True, False]
).reset_index(drop=True)

print("\n=== Category error RATE per model (misses / total in category) ===")
print(rate_by_model_cat.to_string(index=False))

# Top category by error RATE per model (i.e. the category the model is
# proportionally worst at, not just the one with the most raw misses)
top_category_by_rate_per_model = (
    rate_by_model_cat.sort_values(["model", "error_rate"], ascending=[True, False])
    .groupby("model")
    .head(1)
    .reset_index(drop=True)
)

print("\n=== Top category by ERROR RATE per model (pooled across prompts) ===")
print(top_category_by_rate_per_model.to_string(index=False))

# Optional: same breakdown but also split by prompt (rate within
# model x prompt x category)
total_by_model_prompt_cat = (
    long_df.groupby(["model", "prompt", CATEGORY_COL])
    .size()
    .rename("n_total")
    .reset_index()
)
rate_by_model_prompt_cat = miss_counts_by_model_prompt.merge(
    total_by_model_prompt_cat, on=["model", "prompt", CATEGORY_COL], how="right"
)
rate_by_model_prompt_cat["n_misses"] = rate_by_model_prompt_cat["n_misses"].fillna(0)
rate_by_model_prompt_cat["error_rate"] = (
    rate_by_model_prompt_cat["n_misses"] / rate_by_model_prompt_cat["n_total"]
)
rate_by_model_prompt_cat = rate_by_model_prompt_cat.sort_values(
    ["model", "prompt", "error_rate"], ascending=[True, True, False]
).reset_index(drop=True)

# ---------------------------------------------------------------
# 6. Optional: overall accuracy per model (pooled across prompts) for context
# ---------------------------------------------------------------
total_by_model = long_df.groupby("model").size().rename("n_total")
miss_by_model = misses.groupby("model").size().rename("n_miss")
acc_summary = pd.concat([total_by_model, miss_by_model], axis=1).fillna(0)
acc_summary["miss_rate"] = acc_summary["n_miss"] / acc_summary["n_total"]
acc_summary = acc_summary.sort_values("miss_rate")
print("\n=== Overall miss rate per model (pooled across prompts) ===")
print(acc_summary.to_string())

# ---------------------------------------------------------------
# 7. Save results to Excel for easy inspection
# ---------------------------------------------------------------
out_path = press_conf_fed / "model_disagreement_analysis.xlsx"
with pd.ExcelWriter(out_path) as writer:
    misses.to_excel(writer, sheet_name="all_misses", index=False)
    miss_counts_by_model_prompt.to_excel(
        writer, sheet_name="misses_by_model_prompt_cat", index=False
    )
    ranked_by_model.to_excel(writer, sheet_name="misses_by_model_cat", index=False)
    top_category_per_model.to_excel(
        writer, sheet_name="top_category_per_model", index=False
    )
    rate_by_model_cat.to_excel(writer, sheet_name="error_rate_by_model_cat", index=False)
    top_category_by_rate_per_model.to_excel(
        writer, sheet_name="top_category_by_rate", index=False
    )
    rate_by_model_prompt_cat.to_excel(
        writer, sheet_name="error_rate_by_model_prompt_cat", index=False
    )
    acc_summary.to_excel(writer, sheet_name="overall_miss_rate")

print(f"\nSaved detailed results to: {out_path}")

=== Misses by model, prompt, and category ===
            model             prompt       Dissagreement  n_misses
     claude-haiku  p1_analyst_direct     Economic Status       156
     claude-haiku  p1_analyst_direct               Labor        17
     claude-haiku  p1_analyst_direct     Foreign Nations         9
     claude-haiku  p1_analyst_direct Energy/House Prices         7
     claude-haiku  p1_analyst_direct   Fed Expectations          6
     claude-haiku  p1_analyst_direct             General         4
     claude-haiku  p1_analyst_direct        Money Supply         2
     claude-haiku  p1_analyst_direct Dollar Value Change         1
     claude-haiku  p2_analyst_rubric     Economic Status       162
     claude-haiku  p2_analyst_rubric               Labor        16
     claude-haiku  p2_analyst_rubric     Foreign Nations         7
     claude-haiku  p2_analyst_rubric Energy/House Prices         5
     claude-haiku  p2_analyst_rubric   Fed Expectations          5
     claude-haik

ECB Side

In [5]:
# %% [markdown]
# ECB Side — model disagreement analysis + Fed vs ECB comparison
#
# Mirrors the Fed-side analysis: for every (sentence, model, prompt) it flags a
# "miss" when the model's predicted sentiment (D/N/H) != the human `label`, then
# summarizes which topic category (`Dissagreement`) each model gets wrong most —
# in raw counts AND as an error RATE (misses / predictions in that category).
# Then it aligns the analogous Fed/ECB categories and outputs a side-by-side
# comparison.

# %%
import pandas as pd
import numpy as np
from pathlib import Path

# ---- paths (same structure as the Fed side) -------------------------------
press_conf_fed = Path(
    '/Users/lorenzouberti/Desktop/cvbnjkl'
)
press_conf_ecb = Path(
    '/Users/lorenzouberti/Desktop/cvbnjkl'
)

FED_FILE = press_conf_fed / "fed_statements_presconf_WITH_LLM_LABELS_FULL.xlsx"

# Use the 2021+ ECB split so the Fed vs ECB comparison is apples-to-apples
# (the Fed data is 2021+ only). To analyze the full ECB history instead, point
# this at "ECB_statements_presconf_WITH_LLM_LABELS.xlsx".
ECB_FILE = press_conf_ecb / "ECB_statements_presconf_WITH_LLM_LABELS_FULL.xlsx"

TRUE_LABEL_COL = "label"          # ground-truth sentiment (D / N / H)
CATEGORY_COL   = "Dissagreement"  # topic category tag for the sentence

# Map the analogous Fed & ECB categories onto a common label so they line up.
# (Member States/Fragmentation is ECB-only and has no Fed counterpart.)
CANON = {
    'Economic Status':             'Economic Status',
    'Fed Expectations':            'Central Bank Expectations',
    'ECB Expectations':            'Central Bank Expectations',
    'Dollar Value Change':         'Currency Value Change',
    'Euro Value Change':           'Currency Value Change',
    'Foreign Nations':             'Foreign Nations/Trade',
    'Foreign Nations/Trade':       'Foreign Nations/Trade',
    'Energy/House Prices':         'Energy/House Prices',
    'Money Supply':                'Money Supply',
    'Key Words':                   'Key Words',
    'Labor':                       'Labor',
    'General':                     'General',
    'Member States/Fragmentation': 'Member States/Fragmentation',
}
def canon(s):  # normalize (handles e.g. the trailing space in 'Fed Expectations ')
    return CANON.get(str(s).strip(), str(s).strip())


# %%
def analyze(df):
    """Run the disagreement analysis on one labelled dataframe.

    Returns a dict of tidy tables. Every sentence receives one prediction from
    each <model>__<prompt> column, so per-category denominators are
    (n_sentences_in_category * n_model_prompt_columns).
    """
    pred_cols = [c for c in df.columns if "__" in c]
    d = df.dropna(subset=[TRUE_LABEL_COL]).copy()
    d[CATEGORY_COL] = d[CATEGORY_COL].astype(str).str.strip()

    # long/tidy: one row per (sentence, model, prompt)
    recs = []
    for col in pred_cols:
        model, prompt = col.split("__", 1)
        sub = d[[TRUE_LABEL_COL, CATEGORY_COL, col]].rename(columns={col: "pred"}).copy()
        sub["model"], sub["prompt"] = model, prompt
        sub["sentence_id"] = d.index
        recs.append(sub)
    long_df = pd.concat(recs, ignore_index=True).dropna(subset=["pred"])
    long_df["is_miss"] = (
        long_df["pred"].astype(str).str.strip().str.upper()
        != long_df[TRUE_LABEL_COL].astype(str).str.strip().str.upper()
    )
    misses = long_df[long_df["is_miss"]].copy()

    # misses by model x prompt x category, and pooled by model x category
    by_mp_cat = (misses.groupby(["model", "prompt", CATEGORY_COL]).size()
                 .rename("n_misses").reset_index()
                 .sort_values(["model", "prompt", "n_misses"], ascending=[True, True, False]))
    by_m_cat = (misses.groupby(["model", CATEGORY_COL]).size()
                .rename("n_misses").reset_index())
    ranked_by_model = by_m_cat.sort_values(["model", "n_misses"], ascending=[True, False]).reset_index(drop=True)
    top_cat_per_model = ranked_by_model.groupby("model").head(1).reset_index(drop=True)

    # error RATE per model x category (misses / predictions in that category)
    tot_m_cat = long_df.groupby(["model", CATEGORY_COL]).size().rename("n_total").reset_index()
    rate_m_cat = by_m_cat.merge(tot_m_cat, on=["model", CATEGORY_COL], how="right")
    rate_m_cat["n_misses"] = rate_m_cat["n_misses"].fillna(0)
    rate_m_cat["error_rate"] = rate_m_cat["n_misses"] / rate_m_cat["n_total"]
    rate_m_cat = rate_m_cat.sort_values(["model", "error_rate"], ascending=[True, False]).reset_index(drop=True)
    top_rate_per_model = (rate_m_cat.sort_values(["model", "error_rate"], ascending=[True, False])
                          .groupby("model").head(1).reset_index(drop=True))

    # overall miss rate per model (pooled across prompts)
    tot_m = long_df.groupby("model").size().rename("n_total")
    mis_m = misses.groupby("model").size().rename("n_miss")
    overall = pd.concat([tot_m, mis_m], axis=1).fillna(0)
    overall["miss_rate"] = overall["n_miss"] / overall["n_total"]
    overall = overall.sort_values("miss_rate").reset_index()

    # pooled-across-everything category table (used by the comparison)
    n_sent = d.groupby(CATEGORY_COL).size().rename("n_sentences")
    tot_c  = long_df.groupby(CATEGORY_COL).size().rename("n_pred_total")
    mis_c  = misses.groupby(CATEGORY_COL).size().rename("n_pred_miss")
    cat_pooled = pd.concat([n_sent, tot_c, mis_c], axis=1).fillna(0)
    cat_pooled["error_rate"] = cat_pooled["n_pred_miss"] / cat_pooled["n_pred_total"]
    cat_pooled = cat_pooled.reset_index().rename(columns={CATEGORY_COL: "category"})

    overall_all = pd.DataFrame({
        "n_sentences": [len(d)], "n_pred_total": [len(long_df)],
        "n_pred_miss": [len(misses)], "miss_rate": [len(misses) / len(long_df)],
    })
    return dict(long_df=long_df, misses=misses, by_mp_cat=by_mp_cat,
                ranked_by_model=ranked_by_model, top_cat_per_model=top_cat_per_model,
                rate_m_cat=rate_m_cat, top_rate_per_model=top_rate_per_model,
                overall=overall, cat_pooled=cat_pooled, overall_all=overall_all)


# %% [markdown]
# ECB analysis

# %%
ecb_df = pd.read_excel(ECB_FILE)
ecb = analyze(ecb_df)

ecb_out = press_conf_ecb / "ecb_model_disagreement_analysis.xlsx"
with pd.ExcelWriter(ecb_out) as w:
    ecb["misses"].to_excel(w, sheet_name="all_misses", index=False)
    ecb["by_mp_cat"].to_excel(w, sheet_name="misses_by_model_prompt_cat", index=False)
    ecb["ranked_by_model"].to_excel(w, sheet_name="misses_by_model_cat", index=False)
    ecb["top_cat_per_model"].to_excel(w, sheet_name="top_category_per_model", index=False)
    ecb["rate_m_cat"].to_excel(w, sheet_name="error_rate_by_model_cat", index=False)
    ecb["top_rate_per_model"].to_excel(w, sheet_name="top_category_by_rate", index=False)
    ecb["overall"].to_excel(w, sheet_name="overall_miss_rate", index=False)
    (ecb["cat_pooled"].sort_values("error_rate", ascending=False)
        .to_excel(w, sheet_name="category_error_rate_pooled", index=False))
print("Saved ECB analysis to:", ecb_out)

print("\n=== ECB overall miss rate per model ===")
print(ecb["overall"].to_string(index=False))
print("\n=== ECB category error rate (pooled) ===")
print(ecb["cat_pooled"].sort_values("error_rate", ascending=False).to_string(index=False))


# %% [markdown]
# Fed vs ECB comparison

# %%
fed = analyze(pd.read_excel(FED_FILE))

# 1) overall miss rate per model
comp_model = (
    fed["overall"].rename(columns={"n_total": "fed_n", "n_miss": "fed_n_miss", "miss_rate": "fed_miss_rate"})
    .merge(ecb["overall"].rename(columns={"n_total": "ecb_n", "n_miss": "ecb_n_miss", "miss_rate": "ecb_miss_rate"}),
           on="model", how="outer")
)
comp_model["delta_ecb_minus_fed"] = comp_model["ecb_miss_rate"] - comp_model["fed_miss_rate"]
comp_model = comp_model[["model", "fed_n", "fed_miss_rate", "ecb_n", "ecb_miss_rate",
                         "delta_ecb_minus_fed"]].sort_values("ecb_miss_rate").reset_index(drop=True)

# 2) overall dataset miss rate
comp_overall = pd.DataFrame({
    "side": ["Fed", "ECB"],
    "n_sentences":  [int(fed["overall_all"].n_sentences[0]),  int(ecb["overall_all"].n_sentences[0])],
    "n_predictions":[int(fed["overall_all"].n_pred_total[0]), int(ecb["overall_all"].n_pred_total[0])],
    "n_misses":     [int(fed["overall_all"].n_pred_miss[0]),  int(ecb["overall_all"].n_pred_miss[0])],
    "miss_rate":    [float(fed["overall_all"].miss_rate[0]),  float(ecb["overall_all"].miss_rate[0])],
})

# 3) category error rate, aligned via CANON
def cat_table(a):
    t = a["cat_pooled"].copy()
    t["canonical"] = t["category"].map(canon)
    t = t.groupby("canonical").agg(n_sentences=("n_sentences", "sum"),
                                   n_pred_total=("n_pred_total", "sum"),
                                   n_pred_miss=("n_pred_miss", "sum")).reset_index()
    t["error_rate"] = t["n_pred_miss"] / t["n_pred_total"]
    return t
ft, et = cat_table(fed), cat_table(ecb)
comp_cat = (
    ft.rename(columns={"n_sentences": "fed_n_sentences", "error_rate": "fed_error_rate"})[["canonical", "fed_n_sentences", "fed_error_rate"]]
    .merge(et.rename(columns={"n_sentences": "ecb_n_sentences", "error_rate": "ecb_error_rate"})[["canonical", "ecb_n_sentences", "ecb_error_rate"]],
           on="canonical", how="outer")
)
comp_cat["delta_ecb_minus_fed"] = comp_cat["ecb_error_rate"] - comp_cat["fed_error_rate"]
comp_cat = comp_cat.sort_values("ecb_error_rate", ascending=False).reset_index(drop=True)

# 4) top problem category per model, aligned
def toprate(a):
    t = a["rate_m_cat"].copy()
    t["canonical"] = t[CATEGORY_COL].map(canon)
    t = t.groupby(["model", "canonical"]).agg(n_misses=("n_misses", "sum"),
                                              n_total=("n_total", "sum")).reset_index()
    t["error_rate"] = t["n_misses"] / t["n_total"]
    return t.sort_values(["model", "error_rate"], ascending=[True, False]).groupby("model").head(1).reset_index(drop=True)
comp_top = (
    toprate(fed).rename(columns={"canonical": "fed_top_category", "error_rate": "fed_error_rate"})[["model", "fed_top_category", "fed_error_rate"]]
    .merge(toprate(ecb).rename(columns={"canonical": "ecb_top_category", "error_rate": "ecb_error_rate"})[["model", "ecb_top_category", "ecb_error_rate"]],
           on="model", how="outer")
).sort_values("model").reset_index(drop=True)

# 5) per-model x canonical-category error rate (long)
def perm(a, side):
    t = a["rate_m_cat"].copy()
    t["canonical"] = t[CATEGORY_COL].map(canon)
    t = t.groupby(["model", "canonical"]).agg(n_misses=("n_misses", "sum"),
                                              n_total=("n_total", "sum")).reset_index()
    t[f"{side}_error_rate"] = t["n_misses"] / t["n_total"]
    return t[["model", "canonical", f"{side}_error_rate"]]
comp_perm = perm(fed, "fed").merge(perm(ecb, "ecb"), on=["model", "canonical"], how="outer")
comp_perm["delta_ecb_minus_fed"] = comp_perm["ecb_error_rate"] - comp_perm["fed_error_rate"]
comp_perm = comp_perm.sort_values(["model", "canonical"]).reset_index(drop=True)

cmp_out = press_conf_ecb / "fed_vs_ecb_disagreement_comparison.xlsx"
with pd.ExcelWriter(cmp_out) as w:
    comp_overall.to_excel(w, sheet_name="overall_dataset", index=False)
    comp_model.to_excel(w, sheet_name="overall_by_model", index=False)
    comp_cat.to_excel(w, sheet_name="category_error_rate", index=False)
    comp_top.to_excel(w, sheet_name="top_category_per_model", index=False)
    comp_perm.to_excel(w, sheet_name="per_model_category_rate", index=False)
print("Saved comparison to:", cmp_out)

pd.set_option("display.float_format", lambda x: f"{x:.3f}")
print("\n=== OVERALL (pooled across all models & prompts) ===");  print(comp_overall.to_string(index=False))
print("\n=== MISS RATE PER MODEL: Fed vs ECB ===");               print(comp_model.to_string(index=False))
print("\n=== CATEGORY ERROR RATE: Fed vs ECB (aligned) ===");     print(comp_cat.to_string(index=False))
print("\n=== TOP PROBLEM CATEGORY PER MODEL ===");                print(comp_top.to_string(index=False))

Saved ECB analysis to: /Users/lorenzouberti/Desktop/cvbnjkl/ecb_model_disagreement_analysis.xlsx

=== ECB overall miss rate per model ===
            model  n_total  n_miss  miss_rate
deepseek-reasoner    10220    2891      0.283
          gpt-5.1    10220    2953      0.289
    mistral-large    10220    3015      0.295
    claude-sonnet    10220    3132      0.306
     claude-haiku    10220    3136      0.307
       gpt-5-mini    10220    3257      0.319
    mistral-small    10220    3479      0.340
    deepseek-chat    10220    3603      0.353

=== ECB category error rate (pooled) ===
                   category  n_sentences  n_pred_total  n_pred_miss  error_rate
Member States/Fragmentation           87          3480         2026       0.582
                      Labor           81          3240         1367       0.422
               Money Supply           83          3320         1347       0.406
      Foreign Nations/Trade           36          1440          583       0.405
      

In [ ]:
# %% [markdown]
# Do open-weight LLMs understand ECB communication worse than proprietary ones?
#
# Models in this study:
#   Proprietary (US): gpt-5.1, gpt-5-mini (OpenAI) ; claude-haiku, claude-sonnet (Anthropic)
#   Open-weight     : mistral-large, mistral-small (Mistral, EU) ; deepseek-chat, deepseek-reasoner (DeepSeek, CN)
#
# The hypothesis "open models understand ECB *communication* worse" is NOT the same
# as "open models are less accurate on ECB" — a model can simply be weaker overall.
# The clean test is the **Fed -> ECB drop**: does a group lose *more* accuracy when
# moving from Fed to ECB than another group? That is the openness x bank interaction.
# Positive drop-gap for open models = they are disproportionately worse on ECB.
#
# A natural competing angle: Mistral is European (ECB is European), the GPT/Claude
# labs are US (Fed is US). If regional/training proximity matters, European models
# might *gain* relative ground on ECB. The region x bank interaction tests that.

# %%
import numpy as np
import pandas as pd
from pathlib import Path

# statsmodels is optional but used for the formal tests / CIs
try:
    import statsmodels.formula.api as smf
    from statsmodels.stats.proportion import proportion_confint
    HAVE_SM = True
except Exception:
    HAVE_SM = False

BASE = Path(
    '/Users/lorenzouberti/Desktop/cvbnjkl'
)
FED_FILE = BASE / "Fed Side/Excel Sentences Results/fed_statements_presconf_WITH_LLM_LABEL.xlsx"
ECB_FILE = BASE / "ECB Side/Excel Results/ECB_statements_presconf_WITH_LLM_LABELS_2021.xlsx"
OUT_DIR  = BASE / "ECB Side/Excel Results"

TRUE_LABEL_COL, CATEGORY_COL = "label", "Dissagreement"

# ---- model groupings (edit here if you disagree with any label) ------------
PROVIDER = {
    'claude-haiku': 'Anthropic', 'claude-sonnet': 'Anthropic',
    'gpt-5.1': 'OpenAI',         'gpt-5-mini': 'OpenAI',
    'deepseek-chat': 'DeepSeek', 'deepseek-reasoner': 'DeepSeek',
    'mistral-large': 'Mistral',  'mistral-small': 'Mistral',
}
OPENNESS = {'Anthropic': 'Proprietary', 'OpenAI': 'Proprietary',
            'DeepSeek': 'Open-weight',   'Mistral': 'Open-weight'}
REGION   = {'OpenAI': 'US', 'Anthropic': 'US', 'Mistral': 'Europe', 'DeepSeek': 'China'}

# euro-area-specific categories: if the "understanding gap" is real, it should
# concentrate here rather than in generic macro categories.
ECB_SPECIFIC_CATS = {'Member States/Fragmentation', 'ECB Expectations', 'Euro Value Change'}


# %%
def build_long(path, bank):
    """One row per (sentence, model, prompt) with a `correct` flag + group labels."""
    df = pd.read_excel(path)
    pred_cols = [c for c in df.columns if "__" in c]
    d = df.dropna(subset=[TRUE_LABEL_COL]).copy()
    d[CATEGORY_COL] = d[CATEGORY_COL].astype(str).str.strip()
    d["sentence_uid"] = f"{bank}_" + d.index.astype(str)   # unique per sentence within bank

    recs = []
    for col in pred_cols:
        model, prompt = col.split("__", 1)
        sub = d[[TRUE_LABEL_COL, CATEGORY_COL, "sentence_uid", col]].rename(columns={col: "pred"}).copy()
        sub["model"], sub["prompt"], sub["bank"] = model, prompt, bank
        recs.append(sub)
    long = pd.concat(recs, ignore_index=True).dropna(subset=["pred"])
    long["correct"] = (long["pred"].astype(str).str.strip().str.upper()
                       == long[TRUE_LABEL_COL].astype(str).str.strip().str.upper()).astype(int)
    long["provider"] = long["model"].map(PROVIDER)
    long["openness"] = long["provider"].map(OPENNESS)
    long["region"]   = long["provider"].map(REGION)
    long["ecb_specific"] = long[CATEGORY_COL].isin(ECB_SPECIFIC_CATS)
    return long

fed_long = build_long(FED_FILE, "Fed")
ecb_long = build_long(ECB_FILE, "ECB")
long = pd.concat([fed_long, ecb_long], ignore_index=True)
long["bank_ecb"] = (long["bank"] == "ECB").astype(int)
long["is_open"]  = (long["openness"] == "Open-weight").astype(int)


# %%
def acc_by(group_col):
    """Accuracy per <group> x bank, wide, with the Fed->ECB drop."""
    g = (long.groupby([group_col, "bank"])["correct"]
         .agg(accuracy="mean", n="count").reset_index())
    wide = g.pivot(index=group_col, columns="bank", values="accuracy")
    n    = g.pivot(index=group_col, columns="bank", values="n")
    wide.columns = [f"{c}_acc" for c in wide.columns]
    wide["fed_to_ecb_drop"] = wide["Fed_acc"] - wide["ECB_acc"]
    wide["n_fed"], wide["n_ecb"] = n["Fed"], n["ECB"]
    return wide.reset_index().sort_values("ECB_acc", ascending=False)

openness_tab = acc_by("openness")
provider_tab = acc_by("provider")
region_tab   = acc_by("region")
model_tab    = acc_by("model")
model_tab["openness"] = model_tab["model"].map(lambda m: OPENNESS[PROVIDER[m]])
model_tab["provider"] = model_tab["model"].map(PROVIDER)

print("=== Accuracy by OPENNESS x bank (higher ECB_acc = better; larger drop = understands ECB worse) ===")
print(openness_tab.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print("\n=== Accuracy by PROVIDER x bank ===")
print(provider_tab.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print("\n=== Accuracy by REGION x bank (home-advantage check) ===")
print(region_tab.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print("\n=== Accuracy by MODEL x bank ===")
print(model_tab[["model", "openness", "Fed_acc", "ECB_acc", "fed_to_ecb_drop"]]
      .to_string(index=False, float_format=lambda x: f"{x:.3f}"))

# The single headline number: differential Fed->ECB drop (open minus proprietary)
open_drop = float(openness_tab.set_index("openness").loc["Open-weight", "fed_to_ecb_drop"])
prop_drop = float(openness_tab.set_index("openness").loc["Proprietary", "fed_to_ecb_drop"])
print(f"\nOpen-weight Fed->ECB drop = {open_drop:.3f} | Proprietary drop = {prop_drop:.3f} "
      f"| differential (open - prop) = {open_drop - prop_drop:+.3f}")
print("  (positive => open models lose MORE accuracy on ECB => understand ECB comms worse)")


# %% [markdown]
# Formal test 1 — prediction-level logistic regression with the interaction.
# correct ~ open * ECB, SEs clustered by sentence (each sentence is rated many times).
# The `is_open:bank_ecb` coefficient is the hypothesis test: negative & significant
# => open models are disproportionately wrong on ECB. High power, but treats the 8
# models as fixed, so it speaks to *these* models, not "all open models".

# %%
if HAVE_SM:
    m1 = smf.logit("correct ~ is_open * bank_ecb", data=long).fit(
        disp=False, cov_type="cluster", cov_kwds={"groups": long["sentence_uid"]})
    print("\n=== Logit: correct ~ is_open * bank_ecb  (cluster-robust by sentence) ===")
    print(m1.summary().tables[1])
    ic = m1.params.get("is_open:bank_ecb", np.nan)
    ip = m1.pvalues.get("is_open:bank_ecb", np.nan)
    print(f"\nInteraction is_open:bank_ecb = {ic:+.3f} (p = {ip:.4f}) "
          f"-> {'open models disproportionately worse on ECB' if (ic<0 and ip<0.05) else 'no significant disproportionate ECB penalty for open models'}")

    # Region home-advantage: does Europe (Mistral) gain on ECB vs US labs?
    reg = long[long["region"].isin(["US", "Europe"])].copy()
    reg["is_eu"] = (reg["region"] == "Europe").astype(int)
    m2 = smf.logit("correct ~ is_eu * bank_ecb", data=reg).fit(
        disp=False, cov_type="cluster", cov_kwds={"groups": reg["sentence_uid"]})
    rc = m2.params.get("is_eu:bank_ecb", np.nan); rp = m2.pvalues.get("is_eu:bank_ecb", np.nan)
    print(f"\nRegion interaction is_eu:bank_ecb = {rc:+.3f} (p = {rp:.4f}) "
          f"-> {'European model gains on ECB vs US (home advantage)' if (rc>0 and rp<0.05) else 'no significant European home-advantage on ECB'}")
else:
    print("\n[statsmodels not available — skipping logit; `pip install statsmodels`]")


# %% [markdown]
# Formal test 2 — MODEL-LEVEL comparison (the honest inferential unit).
# Only 4 open vs 4 proprietary models, so power is low, but this doesn't pseudo-
# replicate. Compares the 8 per-model Fed->ECB drops between the two groups.

# %%
from scipy import stats
md = model_tab.copy()
open_drops = md.loc[md.openness == "Open-weight", "fed_to_ecb_drop"].values
prop_drops = md.loc[md.openness == "Proprietary", "fed_to_ecb_drop"].values
u, pu = stats.mannwhitneyu(open_drops, prop_drops, alternative="greater")  # open drop > prop drop?
print("\n=== Model-level test: are open models' Fed->ECB drops larger? (n=4 vs 4) ===")
print(f"open drops   = {np.round(open_drops,3)}  (mean {open_drops.mean():.3f})")
print(f"prop drops   = {np.round(prop_drops,3)}  (mean {prop_drops.mean():.3f})")
print(f"Mann-Whitney U={u:.1f}, one-sided p={pu:.3f}  "
      f"(also try ECB accuracy directly:)")
open_ecb = md.loc[md.openness == "Open-weight", "ECB_acc"].values
prop_ecb = md.loc[md.openness == "Proprietary", "ECB_acc"].values
u2, pu2 = stats.mannwhitneyu(open_ecb, prop_ecb, alternative="less")       # open acc < prop acc?
print(f"ECB accuracy: open mean {open_ecb.mean():.3f} vs prop mean {prop_ecb.mean():.3f}; "
      f"Mann-Whitney one-sided p={pu2:.3f}")


# %% [markdown]
# Bootstrap 95% CIs (resample sentences within each bank) for the two contrasts:
# the ECB accuracy gap (prop - open) and the differential Fed->ECB drop.

# %%
def bootstrap_contrasts(long, B=3000, seed=0):
    rng = np.random.default_rng(seed)
    # pre-aggregate to sentence x openness (each group has equal preds/sentence)
    agg = (long.groupby(["bank", "sentence_uid", "openness"])["correct"].mean()
           .unstack("openness").reset_index())
    out = {"ecb_gap_prop_minus_open": [], "diff_drop_open_minus_prop": []}
    fed = agg[agg.bank == "Fed"]; ecb = agg[agg.bank == "ECB"]
    for _ in range(B):
        fb = fed.sample(len(fed), replace=True, random_state=rng.integers(1e9))
        eb = ecb.sample(len(ecb), replace=True, random_state=rng.integers(1e9))
        f_open, f_prop = fb["Open-weight"].mean(), fb["Proprietary"].mean()
        e_open, e_prop = eb["Open-weight"].mean(), eb["Proprietary"].mean()
        out["ecb_gap_prop_minus_open"].append(e_prop - e_open)
        out["diff_drop_open_minus_prop"].append((f_open - e_open) - (f_prop - e_prop))
    res = {}
    for k, v in out.items():
        v = np.array(v)
        res[k] = (v.mean(), np.percentile(v, 2.5), np.percentile(v, 97.5))
    return res

boot = bootstrap_contrasts(long)
print("\n=== Bootstrap 95% CIs (resampling sentences) ===")
for k, (m, lo, hi) in boot.items():
    sig = "significant" if (lo > 0 or hi < 0) else "n.s. (CI spans 0)"
    print(f"  {k}: {m:+.3f}  [{lo:+.3f}, {hi:+.3f}]  {sig}")
print("  ecb_gap_prop_minus_open > 0 => proprietary more accurate than open on ECB")
print("  diff_drop_open_minus_prop > 0 => open models lose more Fed->ECB (understand ECB worse)")


# %% [markdown]
# Where is the gap? Per-ECB-category accuracy, open vs proprietary.
# If the story is really about ECB *communication*, the open deficit should be
# largest on the euro-area-specific categories (flagged), not generic macro ones.

# %%
cat = (ecb_long.groupby([CATEGORY_COL, "openness"])["correct"]
       .agg(acc="mean", n="count").reset_index())
cat_acc = cat.pivot(index=CATEGORY_COL, columns="openness", values="acc")
cat_n   = cat.pivot(index=CATEGORY_COL, columns="openness", values="n")
cat_acc["prop_minus_open"] = cat_acc["Proprietary"] - cat_acc["Open-weight"]
cat_acc["n_per_group"] = (cat_n["Proprietary"] / (len(PROVIDER)//2)).round().astype("Int64")  # ~sentences
cat_acc["ecb_specific"] = cat_acc.index.isin(ECB_SPECIFIC_CATS)
cat_acc = cat_acc.sort_values("prop_minus_open", ascending=False)
print("\n=== ECB per-category accuracy: Proprietary vs Open-weight (gap = prop - open) ===")
print(cat_acc.to_string(float_format=lambda x: f"{x:.3f}"))
print("\nMean open-vs-prop gap on ECB-specific cats:",
      f"{cat_acc.loc[cat_acc.ecb_specific, 'prop_minus_open'].mean():+.3f}",
      "| on generic cats:",
      f"{cat_acc.loc[~cat_acc.ecb_specific, 'prop_minus_open'].mean():+.3f}")


# %% [markdown]
# Save everything + a couple of plots.

# %%
out_xlsx = OUT_DIR / "open_vs_proprietary_ecb_analysis.xlsx"
with pd.ExcelWriter(out_xlsx) as w:
    openness_tab.to_excel(w, sheet_name="by_openness", index=False)
    provider_tab.to_excel(w, sheet_name="by_provider", index=False)
    region_tab.to_excel(w, sheet_name="by_region", index=False)
    model_tab.to_excel(w, sheet_name="by_model", index=False)
    cat_acc.reset_index().to_excel(w, sheet_name="ecb_category_open_vs_prop", index=False)
    pd.DataFrame([{ "contrast": k, "estimate": v[0], "ci_low": v[1], "ci_high": v[2]}
                  for k, v in boot.items()]).to_excel(w, sheet_name="bootstrap_ci", index=False)
print("\nSaved:", out_xlsx)

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    # Plot 1: accuracy by openness x bank
    fig, ax = plt.subplots(figsize=(6, 4))
    piv = long.groupby(["openness", "bank"])["correct"].mean().unstack("bank")[["Fed", "ECB"]]
    piv.plot(kind="bar", ax=ax, rot=0)
    ax.set_ylabel("accuracy"); ax.set_title("Sentiment accuracy: open vs proprietary, Fed vs ECB")
    ax.set_ylim(0, 0.75); ax.legend(title="")
    for c in ax.containers:
        ax.bar_label(c, fmt="%.2f", fontsize=8)
    fig.tight_layout(); fig.savefig(OUT_DIR / "acc_openness_bank.png", dpi=150)

    # Plot 2: per-model Fed vs ECB, colored by openness
    fig2, ax2 = plt.subplots(figsize=(6, 5))
    colors = {"Open-weight": "tab:orange", "Proprietary": "tab:blue"}
    for _, r in model_tab.iterrows():
        ax2.scatter(r.Fed_acc, r.ECB_acc, color=colors[r.openness], s=60)
        ax2.annotate(r.model, (r.Fed_acc, r.ECB_acc), fontsize=7,
                     xytext=(4, 4), textcoords="offset points")
    lim = [min(model_tab.Fed_acc.min(), model_tab.ECB_acc.min()) - 0.02,
           max(model_tab.Fed_acc.max(), model_tab.ECB_acc.max()) + 0.02]
    ax2.plot(lim, lim, "k--", lw=0.8, label="Fed = ECB")
    ax2.set_xlim(lim); ax2.set_ylim(lim)
    ax2.set_xlabel("Fed accuracy"); ax2.set_ylabel("ECB accuracy")
    ax2.set_title("Per-model Fed vs ECB (below dashed line = worse on ECB)")
    from matplotlib.lines import Line2D
    ax2.legend(handles=[Line2D([0], [0], marker="o", color="w", markerfacecolor=c, label=k)
                        for k, c in colors.items()] +
                       [Line2D([0], [0], ls="--", color="k", label="Fed = ECB")], fontsize=8)
    fig2.tight_layout(); fig2.savefig(OUT_DIR / "per_model_fed_vs_ecb.png", dpi=150)
    print("Saved plots: acc_openness_bank.png, per_model_fed_vs_ecb.png")
except Exception as e:
    print("[plotting skipped]", e)

=== Accuracy by OPENNESS x bank (higher ECB_acc = better; larger drop = understands ECB worse) ===
   openness  ECB_acc  Fed_acc  fed_to_ecb_drop  n_fed  n_ecb
Proprietary    0.627    0.639            0.012  12013  16700
Open-weight    0.614    0.610           -0.004  12135  16700

=== Accuracy by PROVIDER x bank ===
 provider  ECB_acc  Fed_acc  fed_to_ecb_drop  n_fed  n_ecb
   OpenAI    0.628    0.644            0.016   6070   8350
Anthropic    0.626    0.634            0.008   5943   8350
  Mistral    0.621    0.626            0.005   6070   8350
 DeepSeek    0.608    0.595           -0.013   6065   8350

=== Accuracy by REGION x bank (home-advantage check) ===
region  ECB_acc  Fed_acc  fed_to_ecb_drop  n_fed  n_ecb
    US    0.627    0.639            0.012  12013  16700
Europe    0.621    0.626            0.005   6070   8350
 China    0.608    0.595           -0.013   6065   8350

=== Accuracy by MODEL x bank ===
            model    openness  Fed_acc  ECB_acc  fed_to_ecb_drop
     